# Verify Matrix Integrity, Basic stats, and Histograms

    1. Shape should be square
    2. diagonal should be equal to 0
    3. Should be symmetric
    4. off diagonal should have no 0 pairs

    Additionally:
    1. Headers and row indices should match *within* a same matrix
    2. Headers (and row indices) should match *between* matrices after removing the segment ID


## 1. Matrix integrity

In [ ]:
import os
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import tensorly as tl
import dask.array as da
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import yeojohnson
import umap


In [ ]:
# get data
folder = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_distances")
files = sorted(folder.glob("distance_*.parquet"))

print(f"Segment {[i for i in range(1,9)]} exist {[os.path.exists(file) for file in files]}")

In [ ]:
# Load all segments and make df for integrity metrics
segment_names = ['PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'MP', 'NS']

all_segments = []
headers = []
row_indices = []
stats_rows = []

for index, file in enumerate(files):
    print(f"Index: {index}. file: {file}")

    df = pd.read_parquet(file)
    df = df.set_index("column00000")
    all_segments.append(df)
    headers.append(df.columns)
    row_indices.append(df.index)

    diagonal = np.diag(df.values)
    upper_tri = np.triu(df.values, k=1)

    stats_rows.append({
        "segment": segment_names[index],
        "rows": df.shape[0],
        "cols": df.shape[1],
        "diagonal_sum": diagonal.sum(),
        "diagonal_min": diagonal.min(),
        "diagonal_max": diagonal.max(),
        "nonzero_diagonal": int(np.sum(diagonal != 0)),
        "symmetric": np.allclose(df.values, df.values.T, atol=1e-10),
        "zero_offdiag_pairs": int(np.sum(upper_tri == 0)),
        "dist_min": df.values.min(),
        "dist_max": df.values.max(),
    })

segment_stats = pd.DataFrame(stats_rows)
segment_stats

### 1.1 Drop NonZero diganoal if present in all segments
    - we can see that some nonzero diagonals are present in all segments. That's a sign of a problematic sample which I will drop from all segment dfs

In [ ]:
# Collect samples with non-zero diagonal per segment
problem_samples = {}

for index, df in enumerate(all_segments):
    seg = segment_names[index]
    diagonal = np.diag(df.values)
    nonzero_mask = diagonal != 0
    bad = df.index[nonzero_mask].tolist()
    if bad:
        problem_samples[seg] = bad
        print(f"{seg}: {len(bad)} problematic sample(s)")

# Parse out segment-specific fields to get comparable keys: (subtype, isolate_id)
def parse_key(sample):
    parts = sample.split("|")
    return (parts[0], parts[4])

# Build a set of parsed keys per segment, then intersect to find common offenders
problem_key_sets = []
for seg, samples in problem_samples.items():
    keys = {parse_key(s) for s in samples}
    problem_key_sets.append(keys)

common_problem_keys = set.intersection(*problem_key_sets) if problem_key_sets else set()
print(f"\nProblematic samples common to ALL affected segments: {len(common_problem_keys)}")

# For easy inspection
common_problems_df = pd.DataFrame(sorted(common_problem_keys), columns=["subtype", "isolate_id"])
common_problems_df

In [ ]:
# Build the set of full index labels to drop per segment
# A row matches if its parsed key is in common_problem_keys
cleaned_segments = []

for index, df in enumerate(all_segments):
    to_drop = [s for s in df.index if parse_key(s) in common_problem_keys]
    df_clean = df.drop(index=to_drop, errors="ignore")
    df_clean = df_clean.drop(columns=to_drop, errors="ignore")  # also drop corresponding columns
    cleaned_segments.append(df_clean)
    print(f"{segment_names[index]}: {df.shape} -> {df_clean.shape}  (dropped {len(to_drop)} samples)")

all_segments = cleaned_segments

### 1.2 Normalization
    We expect some degree of incoherence in FAMSA in both sides of the diagonal. We will normalize the values by taking the average of both sides of the diagonal. We will force the diagonal to be 0.

In [ ]:
# Symmetrize by averaging both sides of the diagonal, then force diagonal to 0
symmetrized_segments = []

for index, df in enumerate(all_segments):
    vals = df.values.copy()
    vals = (vals + vals.T) / 2
    np.fill_diagonal(vals, 0.0)
    df_sym = pd.DataFrame(vals, index=df.index, columns=df.columns)
    symmetrized_segments.append(df_sym)

    # Quick sanity check
    print(f"{segment_names[index]}: symmetric={np.allclose(df_sym.values, df_sym.values.T)}, diag_sum={np.diag(df_sym.values).sum():.6f}")

all_segments = symmetrized_segments

In [ ]:
# Recompute headers and row indices from cleaned/symmetrized segments
headers = [df.columns for df in all_segments]
row_indices = [df.index for df in all_segments]

# Check headers match row indices within each matrix
print("=== Header vs Row Index (within each matrix) ===")
for index, seg in enumerate(segment_names):
    match = all(headers[index] == row_indices[index])
    print(f"  {seg}: {match}")

# Extract virus IDs (field at position 1) without segment info
all_virus_ids = []
for header in headers:
    all_virus_ids.append([h.split("|")[1] for h in header])

# Check virus IDs are consistent across all segments vs segment 1
print(f"\n=== Virus ID consistency (vs {segment_names[0]}) ===")
for index, seg in enumerate(segment_names):
    match = all_virus_ids[0] == all_virus_ids[index]
    print(f"  {seg}: {match}")

In [ ]:

# Check for near-zero distances (per segment) 
for seg_idx, segment in enumerate(all_segments):
    vals = segment.values[np.triu_indices_from(segment.values, k=1)]
    pct_near_zero = (vals < 1e-6).sum() / len(vals) * 100
    
    print(
        f"Segment {seg_idx}: {pct_near_zero:.2f}% near-zero distances, "
        f"min={vals.min():.4f}, median={np.median(vals):.4f}, max={vals.max():.4f}"
    )

In [ ]:
output_folder = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_distances")

for index, df in enumerate(all_segments):
    out_path = output_folder / f"symmetric_distances_{index + 1}.parquet"
    df.to_parquet(out_path)
    print(f"Saved {segment_names[index]} -> {out_path}")

## 2. Basic Stats and Histograms

### 2.1 Stats

In [ ]:
# Load the symmetric parquet files
folder = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_distances")
files = sorted(folder.glob("symmetric_distances_*.parquet"))
segment_names = ['PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'MP', 'NS']

In [ ]:

all_segments = []
for f in files:
    df = pd.read_parquet(f).set_index("column00000") if "column00000" in pd.read_parquet(f).columns else pd.read_parquet(f)
    all_segments.append(df)

# Extract upper triangle (including diagonal) for each segment
triu_indices = np.triu_indices_from(all_segments[0].values, k=0)

stats_rows = []
for i, segment in enumerate(all_segments):
    data = segment.values[triu_indices]
    q25, q75 = np.percentile(data, [25, 75])
    iqr = q75 - q25
    lower_bound = q25 - 1.5 * iqr
    upper_bound = q75 + 1.5 * iqr
    data_nz = data[data != 0]

    row = {
        "segment": segment_names[i],
        "count": len(data),
        "min": data.min(),
        "max": data.max(),
        "range": data.max() - data.min(),
        "mean": data.mean(),
        "median": np.median(data),
        "std": data.std(),
        "variance": data.var(),
    }

    # Percentiles
    for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
        row[f"p{p}"] = np.percentile(data, p)

    row["iqr"] = iqr

    # Zero analysis
    n_zeros = int(np.sum(data == 0))
    row["zeros"] = n_zeros
    row["zeros_pct"] = 100 * n_zeros / len(data)
    row["nonzeros"] = len(data) - n_zeros
    row["nonzeros_pct"] = 100 - row["zeros_pct"]

    # Outliers (IQR method)
    row["outlier_lower_bound"] = lower_bound
    row["outlier_upper_bound"] = upper_bound
    row["outliers_below"] = int(np.sum(data < lower_bound))
    row["outliers_below_pct"] = 100 * row["outliers_below"] / len(data)
    row["outliers_above"] = int(np.sum(data > upper_bound))
    row["outliers_above_pct"] = 100 * row["outliers_above"] / len(data)

    # Non-zero stats
    if len(data_nz) > 0:
        row["nz_min"] = data_nz.min()
        row["nz_max"] = data_nz.max()
        row["nz_mean"] = data_nz.mean()
        row["nz_median"] = np.median(data_nz)

    stats_rows.append(row)

upper_tri_stats = pd.DataFrame(stats_rows).set_index("segment")
upper_tri_stats.T

In [ ]:
output_path = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_per_segment_analysis/upper_tri_stats.csv")
upper_tri_stats.T.to_csv(output_path)
print(f"Saved to {output_path}")

In [ ]:
## 2.2 Histograms by different transforms 
triu_indices = np.triu_indices_from(all_segments[0].values, k=1)

transformations = {
    'Original': [seg.values[triu_indices] for seg in all_segments],
    'Log1p': [np.log1p(seg.values[triu_indices]) for seg in all_segments],
    'Sqrt': [np.sqrt(seg.values[triu_indices]) for seg in all_segments],
    'Inverse': [1 / (seg.values[triu_indices] + 1e-6) for seg in all_segments],
    'Arcsinh': [np.arcsinh(seg.values[triu_indices]) for seg in all_segments],
}

fig, axes = plt.subplots(len(transformations), len(all_segments), figsize=(24, 3.5 * len(transformations)))

for row, (name, data) in enumerate(transformations.items()):
    for col, seg in enumerate(segment_names):
        ax = axes[row, col]
        ax.hist(data[col], bins=80, edgecolor='none', alpha=0.8)
        if row == 0:
            ax.set_title(seg, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(name, fontsize=11, fontweight='bold')
        ax.tick_params(labelsize=8)

fig.suptitle("Distance distributions under different transformations", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

methods = ["mode", "mean", "median"]

# Collect stats per segment
stats_rows = []
for index, df in enumerate(all_segments):
    upper_tri = df.values[np.triu_indices_from(df.values, k=1)]
    stats_rows.append({
        "segment": segment_names[index],
        "mode": stats.mode(upper_tri, keepdims=False).mode,
        "mean": np.mean(upper_tri),
        "median": np.median(upper_tri),
    })

norm_stats_df = pd.DataFrame(stats_rows)

# Compute global stats
global_upper = np.concatenate([df.values[np.triu_indices_from(df.values, k=1)] for df in all_segments])
global_row = {
    "segment": "GLOBAL",
    "mode": stats.mode(global_upper, keepdims=False).mode,
    "mean": np.mean(global_upper),
    "median": np.median(global_upper),
}
norm_stats_df = pd.concat([norm_stats_df, pd.DataFrame([global_row])], ignore_index=True)

# Normalize with each method and plot
normalized_results = {}  # key: method name, value: list of normalized dfs

for method in methods:
    normalized_segments = []

    fig, axes = plt.subplots(len(all_segments), 2, figsize=(14, 4 * len(all_segments)))
    fig.suptitle(f"Normalization by {method}", fontsize=16, y=1.01)

    for index, df in enumerate(all_segments):
        upper_tri = df.values[np.triu_indices_from(df.values, k=1)]
        divisor = norm_stats_df.loc[norm_stats_df["segment"] == segment_names[index], method].values[0]

        vals = df.values / divisor if divisor != 0 else df.values.copy()
        np.fill_diagonal(vals, 0.0)
        df_norm = pd.DataFrame(vals, index=df.index, columns=df.columns)
        normalized_segments.append(df_norm)

        # Raw
        axes[index, 0].hist(upper_tri, bins=100, edgecolor="none", alpha=0.8)
        axes[index, 0].axvline(divisor, color="red", linestyle="--", label=f"{method}={divisor:.4f}")
        axes[index, 0].set_title(f"{segment_names[index]} — Raw")
        axes[index, 0].set_xlabel("Distance")
        axes[index, 0].legend()

        # Normalized
        upper_norm = vals[np.triu_indices_from(vals, k=1)]
        axes[index, 1].hist(upper_norm, bins=100, edgecolor="none", alpha=0.8, color="orange")
        axes[index, 1].axvline(1.0, color="red", linestyle="--", label=f"{method}=1.0")
        axes[index, 1].set_title(f"{segment_names[index]} — {method.capitalize()}-normalized")
        axes[index, 1].set_xlabel("Normalized distance")
        axes[index, 1].legend()

    plt.tight_layout()
    plt.show()

    normalized_results[method] = normalized_segments

norm_stats_df